# Week 2 (v2): inside the model first, then realistic calls, a second detector and the shortcut test

This notebook extends the week-1 pilot and answers the reviewer's ranked changes. It **reuses your week-1 Drive folder** (`fyp_pilot`): the audio subset, both SSL-AASIST checkpoints and every score already computed.

**Order (most important first, so a short session still gives the key result)**
- **Steps 7–9 · Inside the model, checkpoint A (reviewer change 8).** Per XLS-R layer: how far the call moves genuine and fake speech along the clean real→fake axis, and whether a probe can still separate them. Then a training-free fix: remove the call's drift at one layer (layer mean alignment, label-free) or align its statistics (CORAL), fitted on half the speakers. It is compared with threshold recalibration, a random direction, the same fix fitted on *other* codecs, a reverse patch (add the drift to clean audio), and the cost on clean audio.
- **Step 10 · Every detector on every stage.** Adds an AMR-NB rate sweep (12.2 / 7.4 / 5.9 / 4.75), AMR-WB 12.65 and 23.85 and Opus wideband 16/24 kbit/s (realistic VoLTE/VoIP), Opus with native concealment, bursty packet loss, WebRTC processing, a chained call, the shortcut controls (loudness-normalised, silence-removed), and AASIST as a second architecture. It also saves both output logits so the log-likelihood ratio can be checked.
- **Step 11–12 · Results**, including 95% confidence intervals for every change in EER, per-attack results, and the ceiling check (do fakes "not move" only because they are already at the bottom?).
- **Step 13 · Optional ElevenLabs/Respeecher fakes.**
- **Step 14 · Summary to send to Claude.**

**How to run:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**, and keep the tab open. About **2.5–3 hours** in total; steps 7–9 finish in the first ~45 minutes. Everything saves to Drive and resumes: if the session drops, just Run all again.

In [ ]:
#@title 1 · Settings and Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, glob
CFG = dict(
    SEED=1234, BATCH=16,
    OUT='/content/drive/MyDrive/fyp_pilot',      # the week-1 folder
    WORK='/content/work',
    RUN_DETECTORS=['A', 'B', 'AASIST'],          # A = Best_LA_model_for_DF.pth, B = LA_model.pth, AASIST = Jung et al.
    INSIDE_CKPTS=['A'],                          # steps 7-9; add 'B' only if you have spare time
    INSIDE_LAYERS=[5, 11, 17, 23],               # layers (0-23) used for the test-time fix in step 9
    RESCORE_FOR_LLR=True,                        # re-score week-1 stages once to also save the 'fake' logit
    # Optional: folder in your Drive with the Mendeley ElevenLabs/Respeecher clips (unzipped). '' = skip.
    ELEVEN_DIR='',
)
for d in (CFG['WORK'], f"{CFG['OUT']}/figures", f"{CFG['OUT']}/week2"):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{CFG['OUT']}/subset.csv"), 'Week-1 folder not found: run the week-1 notebook first, or fix CFG["OUT"].'
print(json.dumps(CFG, indent=1))

In [ ]:
#@title 2 · Install packages and codecs
!pip -q install transformers soundfile pandas matplotlib scikit-learn webrtc-noise-gain
!apt-get -qq install -y opus-tools > /dev/null
import subprocess, re, shutil
def encoders(ff='ffmpeg'):
    out = subprocess.run([ff, '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
    return {n for n in ('libgsm', 'libopus', 'pcm_mulaw', 'libopencore_amrnb', 'libvo_amrwbenc') if re.search(rf'\s{n}\s', out)}
FFMPEG, enc = 'ffmpeg', encoders()
FFMPEG_GSM = 'ffmpeg' if 'libgsm' in enc else None          # Colab's own ffmpeg has GSM
if 'libopencore_amrnb' not in enc:                          # ... but not AMR: use the static build, as in week 1
    if not os.path.exists('/content/ffs/ffmpeg'):
        !wget -q -O /content/ffs.tar.xz https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz
        !mkdir -p /content/ffs && tar -xf /content/ffs.tar.xz -C /content/ffs --strip-components=1
    if os.path.exists('/content/ffs/ffmpeg') and 'libopencore_amrnb' in encoders('/content/ffs/ffmpeg'):
        FFMPEG, enc = '/content/ffs/ffmpeg', encoders('/content/ffs/ffmpeg')
HAVE_OPUS_TOOLS = shutil.which('opusenc') is not None and shutil.which('opusdec') is not None
try:
    import webrtc_noise_gain; HAVE_WEBRTC = True
except Exception as e:
    HAVE_WEBRTC = False; print('WebRTC processing not available:', e)
print('ffmpeg:', FFMPEG, '| encoders:', sorted(enc), '| GSM via:', FFMPEG_GSM)
print('opus-tools:', HAVE_OPUS_TOOLS, '| WebRTC:', HAVE_WEBRTC, '| AMR-WB:', 'libvo_amrwbenc' in enc)
assert 'libopencore_amrnb' in enc, 'AMR-NB encoder missing: the static ffmpeg download failed. Rerun this cell.'

In [ ]:
#@title 3 · Code and checkpoints (SSL-AASIST from your Drive, AASIST from GitHub)
REPO = '/content/SSL_Anti-spoofing'
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/TakHemlata/SSL_Anti-spoofing {REPO}
AREPO = '/content/aasist'
if not os.path.exists(AREPO):
    !git clone -q --depth 1 https://github.com/clovaai/aasist {AREPO}
commit = lambda r: subprocess.run(['git', '-C', r, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
COMMITS = {'SSL_Anti-spoofing': commit(REPO), 'aasist': commit(AREPO)}
cks = sorted(glob.glob(f"{CFG['OUT']}/checkpoints/**/*.pth", recursive=True))
CK = {'A': next((p for p in cks if os.path.basename(p) == 'Best_LA_model_for_DF.pth'), None),
      'B': next((p for p in cks if os.path.basename(p) == 'LA_model.pth'), None)}
print('commits:', COMMITS)
print('SSL-AASIST checkpoints:', {k: (os.path.basename(v) if v else None) for k, v in CK.items()})
assert CK['A'], 'Best_LA_model_for_DF.pth not found under fyp_pilot/checkpoints (it was downloaded in week 1).'
if not CK['B'] and 'B' in CFG['RUN_DETECTORS']:
    print('LA_model.pth not found: checkpoint B will be skipped.'); CFG['RUN_DETECTORS'].remove('B')

In [ ]:
#@title 4 · The week-1 audio subset (from Drive)
import numpy as np, pandas as pd
SUB = f"{CFG['OUT']}/subset"
subset = pd.read_csv(f"{CFG['OUT']}/subset.csv")
subset['path'] = SUB + '/' + subset.utt + '.flac'
missing = [p for p in subset.path if not os.path.exists(p)]
assert not missing, f'{len(missing)} subset files missing, e.g. {missing[:3]}'
EV, DV = subset[subset.part == 'eval'].reset_index(drop=True), subset[subset.part == 'dev'].reset_index(drop=True)
print(subset.groupby(['part', 'label']).size().rename('n'))
print('eval speakers:', EV.speaker.nunique())

In [ ]:
%%writefile /content/pilot_lib.py
# ---------------------------------------------------------------------------
# Pilot library: call-stage simulator, alignment, SSL-AASIST loader, metrics.
# Kept in one file so it can be tested outside Colab and pasted into the notebook.
# ---------------------------------------------------------------------------
import os, re, sys, types, json, zlib, shutil, tempfile, subprocess
import numpy as np
import soundfile as sf
from scipy.signal import butter, sosfiltfilt, correlate

SR = 16000
CUT = 64600          # SSL-AASIST input length (~4.04 s), as in Tak et al.'s code


# ----------------------------- audio helpers ------------------------------
def tak_pad(x, max_len=CUT):
    """Exactly Tak et al.'s pad(): truncate, or tile short clips."""
    if len(x) >= max_len:
        return x[:max_len]
    reps = int(max_len / len(x)) + 1
    return np.tile(x, reps)[:max_len]


def _run(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + '\n' + r.stderr.decode(errors='ignore')[-800:])


def ffmpeg_roundtrip(x, enc_args, codec_sr=8000, ext='wav', ffmpeg='ffmpeg'):
    """16 kHz float -> (resample to codec_sr) -> encode -> decode -> 16 kHz float."""
    with tempfile.TemporaryDirectory() as d:
        inp, mid, out = f'{d}/in.wav', f'{d}/mid.{ext}', f'{d}/out.wav'
        sf.write(inp, np.clip(x, -1, 1), SR, subtype='PCM_16')
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', inp, '-ac', '1', '-ar', str(codec_sr)] + enc_args + [mid])
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', mid, '-ac', '1', '-ar', str(SR), '-c:a', 'pcm_f32le', out])
        y, _ = sf.read(out, dtype='float32')
    return y


def align_to(ref, y, max_lag=1600):
    """Remove codec delay: find the lag that best matches ref, shift y, trim/pad to len(ref)."""
    n = min(len(ref), len(y), 2 * SR)
    a, b = ref[:n].astype(np.float64), y[:n].astype(np.float64)
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        lag = 0
    else:
        c = correlate(b, a, mode='full', method='fft')
        lags = np.arange(-n + 1, n)
        keep = np.abs(lags) <= max_lag
        lag = int(lags[keep][np.argmax(c[keep])])
    if lag > 0:
        y = y[lag:]
    elif lag < 0:
        y = np.concatenate([np.zeros(-lag, dtype=y.dtype), y])
    if len(y) < len(ref):
        y = np.concatenate([y, np.zeros(len(ref) - len(y), dtype=y.dtype)])
    return y[:len(ref)].astype(np.float32), lag


def bandpass(x, lo=300, hi=3400, order=6):
    sos = butter(order, [lo, hi], btype='bandpass', fs=SR, output='sos')
    return sosfiltfilt(sos, x).astype(np.float32)   # zero-phase: no delay


def rms_normalise(x, target_dbfs=-26.0):
    rms = np.sqrt(np.mean(x ** 2)) + 1e-12
    y = x * (10 ** (target_dbfs / 20) / rms)
    peak = np.max(np.abs(y))
    return (y / peak * 0.99 if peak > 0.99 else y).astype(np.float32)


def frame_loss(x, utt_id, rate, seed, frame=320, fade=40):
    """Drop 20 ms frames at random (seeded per utterance) and conceal by repeating
    the previous frame, attenuated, with short cross-fades. A simplified packet-loss
    concealment, documented as such; not the G.711 Appendix I algorithm."""
    rng = np.random.default_rng((zlib.crc32(utt_id.encode()) ^ seed) & 0xFFFFFFFF)
    y = x.copy()
    n = len(x) // frame
    lost = rng.random(n) < rate
    ramp = np.linspace(0, 1, fade, dtype=np.float32)
    run = 0
    for i in range(1, n):
        s = i * frame
        if lost[i]:
            run += 1
            seg = y[s - frame:s] * (0.7 ** run)          # repeat the last frame, attenuated
            seg[:fade] = y[s - 1] * (1 - ramp) + seg[:fade] * ramp   # smooth join, no click
            y[s:s + frame] = seg
        else:
            if run:   # fade back in from the concealed frame
                y[s:s + fade] = y[s - fade:s] * (1 - ramp) + x[s:s + fade] * ramp
            run = 0
    return y, float(lost.mean())


# ----------------------------- conditions ---------------------------------
def available_encoders(ffmpeg='ffmpeg'):
    out = subprocess.run([ffmpeg, '-hide_banner', '-encoders'], stdout=subprocess.PIPE, stderr=subprocess.PIPE).stdout.decode()
    return {name for name in ('libgsm', 'libopus', 'pcm_mulaw', 'libopencore_amrnb', 'libvo_amrwbenc') if re.search(rf'\s{name}\s', out)}


def make_conditions(encoders, seed, ffmpeg='ffmpeg'):
    """Returns {name: (description, fn(x, utt_id) -> (y, info))}. Every output is
    16 kHz, the same length as the input and sample-aligned with it."""
    C = {}
    C['clean'] = ('No processing', lambda x, u: (x, {}))
    C['clean_rms'] = ('Clean, loudness-normalised to -26 dBFS (loudness/AGC control)',
                      lambda x, u: (rms_normalise(x), {}))
    C['bandpass_300_3400'] = ('Telephone band 300-3,400 Hz, zero-phase filter (band-limit control)',
                              lambda x, u: (bandpass(x), {}))

    def coded(args, ext, sr=8000):
        def fn(x, u):
            y, lag = align_to(x, ffmpeg_roundtrip(x, args, sr, ext, ffmpeg))
            return y, {'lag': lag}
        return fn

    C['nb_resample'] = ('16 -> 8 -> 16 kHz resampling only (control for all 8 kHz codecs)',
                        coded(['-c:a', 'pcm_s16le'], 'wav'))
    if 'pcm_mulaw' in encoders:
        C['g711u'] = ('G.711 mu-law, 64 kbps', coded(['-c:a', 'pcm_mulaw'], 'wav'))

        def g711_loss(x, u):
            y, info = C['g711u'][1](x, u)
            y, lost = frame_loss(y, u, 0.10, seed)
            return y, dict(info, lost=lost)
        C['g711u_loss10'] = ('G.711 mu-law with 10% 20 ms frame loss, repetition concealment', g711_loss)
    if 'libgsm' in encoders:
        C['gsm_fr'] = ('GSM full-rate, 13 kbps (2G mobile)', coded(['-c:a', 'libgsm'], 'gsm'))
    if 'libopencore_amrnb' in encoders:
        C['amrnb_12k2'] = ('AMR-NB 12.2 kbps (mobile)', coded(['-c:a', 'libopencore_amrnb', '-b:a', '12.2k'], 'amr'))
        C['amrnb_4k75'] = ('AMR-NB 4.75 kbps (mobile, low rate)', coded(['-c:a', 'libopencore_amrnb', '-b:a', '4.75k'], 'amr'))
    if 'libopus' in encoders:
        C['opus_nb_8k'] = ('Opus narrowband, 8 kbps, VoIP mode',
                           coded(['-c:a', 'libopus', '-b:a', '8k', '-application', 'voip'], 'opus'))
    return C


# ----------------------------- metrics ------------------------------------
def det_curve(tar, non):
    """Identical to Tak et al.'s compute_det_curve (target = bona fide)."""
    n = tar.size + non.size
    s = np.concatenate([tar, non]); l = np.concatenate([np.ones(tar.size), np.zeros(non.size)])
    idx = np.argsort(s, kind='mergesort'); l = l[idx]
    tsum = np.cumsum(l); nsum = non.size - (np.arange(1, n + 1) - tsum)
    frr = np.concatenate([[0], tsum / tar.size]); far = np.concatenate([[1], nsum / non.size])
    thr = np.concatenate([[s[idx[0]] - 0.001], s[idx]])
    return frr, far, thr


def eer(tar, non):
    frr, far, thr = det_curve(tar, non)
    i = np.argmin(np.abs(frr - far))
    return float(np.mean([frr[i], far[i]])), float(thr[i])


def rates_at(tar, non, t):
    """Bona fide rejected (false alarm on a real caller) and spoof accepted (miss), at threshold t.
    A score >= t is accepted as bona fide."""
    return float(np.mean(tar < t)), float(np.mean(non >= t))


def dcf(p_fa_bona, p_miss_spoof, p_spoof, c_rej_bona=1.0, c_acc_spoof=10.0):
    """Normalised detection cost. 'Rejecting bona fide' and 'accepting spoof' are the two errors.
    Defaults follow ASVspoof 5 (C=1 and 10, P_spoof=0.05)."""
    cost = c_rej_bona * (1 - p_spoof) * p_fa_bona + c_acc_spoof * p_spoof * p_miss_spoof
    return float(cost / min(c_rej_bona * (1 - p_spoof), c_acc_spoof * p_spoof))


def min_dcf(tar, non, p_spoof, c1=1.0, c2=10.0):
    frr, far, _ = det_curve(tar, non)
    return float(min(dcf(fr, fa, p_spoof, c1, c2) for fr, fa in zip(frr, far)))


def boot_ci(d, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    m = [rng.choice(d, d.size, replace=True).mean() for _ in range(n)]
    return float(np.mean(d)), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


PRIORS = {'asv5': (0.05, 1.0, 10.0), 'balanced': (0.5, 1.0, 1.0)}


def summarise(scores, clean, thr):
    """scores/clean: DataFrames indexed by utt with columns score, label(1=bona fide), correct_clean."""
    out = {}
    for subset, mask in (('all', np.ones(len(scores), bool)), ('clean_correct', scores['correct_clean'].values)):
        s = scores[mask]; c = clean.loc[s.index]
        tar, non = s[s.label == 1].score.values, s[s.label == 0].score.values
        if tar.size < 2 or non.size < 2:          # e.g. an empty clean-correct subset
            out[subset] = {'eer': np.nan, 'fa_bona_at_thr': np.nan, 'miss_spoof_at_thr': np.nan,
                           'n_bona': int(tar.size), 'n_spoof': int(non.size)}
            continue
        e, _ = eer(tar, non)
        fa, miss = rates_at(tar, non, thr)
        r = {'eer': e, 'fa_bona_at_thr': fa, 'miss_spoof_at_thr': miss, 'hter_at_thr': (fa + miss) / 2}
        for k, (p, c1, c2) in PRIORS.items():
            r[f'minDCF_{k}'] = min_dcf(tar, non, p, c1, c2)
            r[f'actDCF_{k}'] = dcf(fa, miss, p, c1, c2)
            r[f'calib_loss_{k}'] = r[f'actDCF_{k}'] - r[f'minDCF_{k}']
        for lab, name in ((1, 'bona'), (0, 'spoof')):
            d = (s[s.label == lab].score.values - c[c.label == lab].score.values)
            m, lo, hi = boot_ci(d)
            r[f'shift_{name}_mean'], r[f'shift_{name}_lo'], r[f'shift_{name}_hi'] = m, lo, hi
        r['n_bona'], r['n_spoof'] = int(tar.size), int(non.size)
        out[subset] = r
    return out


def decision(row_clean, row):
    """Pre-registered pilot rule: clean EER <= 5% AND (EER rises >= 10 points OR
    real callers rejected at the transferred threshold rise >= 15 points)."""
    ok_clean = row_clean['eer'] <= 0.05
    d_eer = row['eer'] - row_clean['eer']
    d_fa = row['fa_bona_at_thr'] - row_clean['fa_bona_at_thr']
    counts = ok_clean and (d_eer >= 0.10 or d_fa >= 0.15)
    if not counts:
        kind = 'no_effect'
    elif d_eer >= 0.10:
        kind = 'ranking_breaks'
    else:
        kind = 'threshold_only'
    return counts, kind, d_eer, d_fa


# ----------------------------- model loader -------------------------------
def xlsr_config(n_layers=24, conv_bias=True):
    from transformers import Wav2Vec2Config
    return Wav2Vec2Config(
        hidden_size=1024, num_hidden_layers=n_layers, num_attention_heads=16, intermediate_size=4096,
        hidden_act='gelu', hidden_dropout=0.0, activation_dropout=0.0, attention_dropout=0.0,
        feat_proj_dropout=0.0, final_dropout=0.0, layerdrop=0.0, layer_norm_eps=1e-5,
        feat_extract_norm='layer', feat_extract_activation='gelu', conv_dim=(512,) * 7,
        conv_stride=(5, 2, 2, 2, 2, 2, 2), conv_kernel=(10, 3, 3, 3, 3, 2, 2), conv_bias=conv_bias,
        num_conv_pos_embeddings=128, num_conv_pos_embedding_groups=16, do_stable_layer_norm=True,
        apply_spec_augment=False, mask_time_prob=0.0, mask_feature_prob=0.0)


IGNORED_FS_PREFIXES = ('quantizer.', 'project_q.', 'final_proj.', 'project_inp.', 'target_glu.', 'input_quantizer.', 'mask_emb')


def map_fairseq_to_hf(sd_fs, hf_keys):
    """Rename fairseq wav2vec 2.0 parameter names to Hugging Face Wav2Vec2Model names.
    Module names checked against the fairseq commit bundled in Tak et al.'s repository."""
    hf_keys = set(hf_keys)
    param_wn = any('parametrizations.weight.original0' in k for k in hf_keys)
    out, unused = {}, []
    for k, v in sd_fs.items():
        nk = None
        if m := re.fullmatch(r'feature_extractor\.conv_layers\.(\d+)\.0\.(weight|bias)', k):
            nk = f'feature_extractor.conv_layers.{m[1]}.conv.{m[2]}'
        elif m := re.fullmatch(r'feature_extractor\.conv_layers\.(\d+)\.2\.1\.(weight|bias)', k):
            nk = f'feature_extractor.conv_layers.{m[1]}.layer_norm.{m[2]}'
        elif k in ('layer_norm.weight', 'layer_norm.bias'):
            nk = 'feature_projection.' + k
        elif m := re.fullmatch(r'post_extract_proj\.(weight|bias)', k):
            nk = f'feature_projection.projection.{m[1]}'
        elif k == 'encoder.pos_conv.0.bias':
            nk = 'encoder.pos_conv_embed.conv.bias'
        elif k == 'encoder.pos_conv.0.weight_g':
            nk = 'encoder.pos_conv_embed.conv.' + ('parametrizations.weight.original0' if param_wn else 'weight_g')
        elif k == 'encoder.pos_conv.0.weight_v':
            nk = 'encoder.pos_conv_embed.conv.' + ('parametrizations.weight.original1' if param_wn else 'weight_v')
        elif m := re.fullmatch(r'encoder\.layers\.(\d+)\.(.+)', k):
            rest = (m[2].replace('self_attn_layer_norm', 'layer_norm').replace('self_attn.', 'attention.')
                    .replace('fc1.', 'feed_forward.intermediate_dense.').replace('fc2.', 'feed_forward.output_dense.'))
            nk = f'encoder.layers.{m[1]}.{rest}'
        elif k in ('encoder.layer_norm.weight', 'encoder.layer_norm.bias'):
            nk = k
        elif k == 'mask_emb':
            nk = 'masked_spec_embed'
        if nk is None or nk not in hf_keys:
            unused.append(k)
        else:
            out[nk] = v
    missing = sorted(hf_keys - set(out))
    return out, unused, missing


def build_ssl_aasist(repo_dir, ckpt_path, device='cpu'):
    """Tak et al.'s SSL-AASIST with the fairseq front-end replaced by an identical
    Hugging Face Wav2Vec2Model, so no fairseq install is needed. Loads strictly."""
    import torch, torch.nn as nn
    from transformers import Wav2Vec2Model
    sd = torch.load(ckpt_path, map_location='cpu')
    if isinstance(sd, dict) and 'state_dict' in sd and isinstance(sd['state_dict'], dict):
        sd = sd['state_dict']
    sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}
    pre = 'ssl_model.model.'
    fs = {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre)}
    n_layers = 1 + max(int(m[1]) for k in fs if (m := re.match(r'encoder\.layers\.(\d+)\.', k)))
    conv_bias = 'feature_extractor.conv_layers.0.0.bias' in fs
    cfg = xlsr_config(n_layers, conv_bias)

    class HFSSL(nn.Module):
        def __init__(self, device):
            super().__init__()
            self.model = Wav2Vec2Model(cfg)
            self.out_dim = cfg.hidden_size
            self.device = device

        def extract_feat(self, x, output_hidden_states=False):
            if x.ndim == 3:
                x = x[:, :, 0]
            o = self.model(x, output_hidden_states=output_hidden_states)
            if output_hidden_states:
                self.last_hidden_states = o.hidden_states
            return o.last_hidden_state

    if 'fairseq' not in sys.modules:                 # model.py imports fairseq at the top; never used here
        sys.modules['fairseq'] = types.ModuleType('fairseq')
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    import model as tak
    tak.SSLModel = HFSSL
    net = tak.Model(None, device)
    hf_keys = list(net.ssl_model.model.state_dict().keys())
    mapped, unused, missing = map_fairseq_to_hf(fs, hf_keys)
    bad_unused = [k for k in unused if not k.startswith(IGNORED_FS_PREFIXES)]
    missing = [k for k in missing if k != 'masked_spec_embed']
    if bad_unused or missing:
        raise RuntimeError(f'Weight mapping failed.\nUnmapped checkpoint keys: {bad_unused[:20]}\nMissing model keys: {missing[:20]}')
    full = {k: v for k, v in sd.items() if not k.startswith(pre)}
    full.update({pre + k: v for k, v in mapped.items()})
    if 'masked_spec_embed' in hf_keys and 'masked_spec_embed' not in mapped:   # only exists in some versions
        full[pre + 'masked_spec_embed'] = net.ssl_model.model.masked_spec_embed.detach()
    net.load_state_dict(full, strict=True)
    report = {'xlsr_layers': n_layers, 'conv_bias': conv_bias, 'ssl_params_loaded': len(mapped),
              'ignored_pretraining_params': len(unused), 'backend_params_loaded': len(sd) - len(fs)}
    return net.to(device).eval(), report


# ===========================================================================
# Week 2 additions: shortcut controls, more realistic channels, a second
# architecture (AASIST) and hooks for looking inside the XLS-R front end.
# ===========================================================================
def trim_bounds(x, frame=320, rel_db=-35.0, margin=1600):
    """Start/end sample of the speech region: frames within rel_db of the loudest
    20 ms frame, padded by 100 ms. Computed on the CLEAN copy and applied to the
    call copy too, so the pair stays aligned (silence-shortcut control)."""
    n = max(1, len(x) // frame)
    e = np.array([np.mean(x[i * frame:(i + 1) * frame] ** 2) for i in range(n)]) + 1e-12
    db = 10 * np.log10(e / e.max())
    idx = np.where(db > rel_db)[0]
    if idx.size == 0:
        return 0, len(x)
    return max(0, idx[0] * frame - margin), min(len(x), (idx[-1] + 1) * frame + margin)


def opus_plc(x, loss_pct, bitrate_k=8, opusenc='opusenc', opusdec='opusdec', ffmpeg='ffmpeg'):
    """Opus narrowband speech at bitrate_k kbit/s with real Opus packet-loss concealment
    (opusdec --packet-loss). opusdec draws losses from an unseeded rand(), so the loss
    pattern is deterministic and identical for every file: documented, reproducible."""
    with tempfile.TemporaryDirectory() as d:
        sf.write(f'{d}/in.wav', np.clip(x, -1, 1), SR, subtype='PCM_16')
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', f'{d}/in.wav', '-ac', '1', '-ar', '8000', f'{d}/in8.wav'])
        _run([opusenc, '--quiet', '--bitrate', str(bitrate_k), '--speech', '--framesize', '20', f'{d}/in8.wav', f'{d}/a.opus'])
        _run([opusdec, '--quiet', '--rate', str(SR), '--packet-loss', str(loss_pct), f'{d}/a.opus', f'{d}/o.wav'])
        y, _ = sf.read(f'{d}/o.wav', dtype='float32')
    if y.ndim > 1:
        y = y[:, 0]
    return y


def webrtc_apm(x, noise_suppression=2, auto_gain_dbfs=3):
    """WebRTC noise suppression + automatic gain control (+ its high-pass filter), as phones
    and VoIP apps apply before encoding. 10 ms chunks of 16 kHz int16."""
    from webrtc_noise_gain import AudioProcessor
    ap = AudioProcessor(auto_gain_dbfs, noise_suppression)
    pcm = (np.clip(x, -1, 1) * 32767).astype(np.int16)
    n = len(pcm) // 160 * 160
    out = bytearray()
    for i in range(0, n, 160):
        out += ap.Process10ms(pcm[i:i + 160].tobytes()).audio
    y = np.frombuffer(bytes(out), dtype=np.int16).astype(np.float32) / 32768
    return np.concatenate([y, np.zeros(len(x) - len(y), np.float32)])


def _coded(args, ext, sr, ffmpeg):
    def fn(x, u):
        y, lag = align_to(x, ffmpeg_roundtrip(x, args, sr, ext, ffmpeg))
        return y, {'lag': lag}
    return fn


def frame_loss_ge(x, utt_id, mean_loss, mean_burst, seed, frame=320, fade=40):
    """Bursty 20 ms frame loss from a two-state Gilbert-Elliott chain (seeded per utterance), with the
    same repetition concealment as frame_loss. mean_loss = long-run loss rate, mean_burst = frames per burst."""
    rng = np.random.default_rng((zlib.crc32(utt_id.encode()) ^ (seed + 7)) & 0xFFFFFFFF)
    p_bg = 1.0 / mean_burst                       # bad -> good
    p_gb = mean_loss * p_bg / (1 - mean_loss)     # good -> bad
    n = len(x) // frame
    lost, bad = np.zeros(n, bool), False
    for i in range(n):
        bad = (rng.random() < p_gb) if not bad else (rng.random() >= p_bg)
        lost[i] = bad
    lost[0] = False
    y = x.copy(); ramp = np.linspace(0, 1, fade, dtype=np.float32); run = 0
    for i in range(1, n):
        s = i * frame
        if lost[i]:
            run += 1
            seg = y[s - frame:s] * (0.7 ** run)
            seg[:fade] = y[s - 1] * (1 - ramp) + seg[:fade] * ramp
            y[s:s + frame] = seg
        else:
            if run:
                y[s:s + fade] = y[s - fade:s] * (1 - ramp) + x[s:s + fade] * ramp
            run = 0
    return y, float(lost.mean())


def make_conditions_v2(encoders, seed, ffmpeg='ffmpeg', ffmpeg_gsm=None, have_opus_tools=False, have_webrtc=False):
    """Week-2 stages. Returns {name: (description, fn(x, utt_id) -> (y, info))}.
    Shortcut controls (names with __rms / __trim) are added later, per damaging stage."""
    base = make_conditions(encoders, seed, ffmpeg)
    C = {}
    if ffmpeg_gsm and 'libgsm' in available_encoders(ffmpeg_gsm):
        def gsm(x, u):
            y, lag = align_to(x, ffmpeg_roundtrip(x, ['-c:a', 'libgsm'], 8000, 'gsm', ffmpeg_gsm))
            return y, {'lag': lag}
        C['gsm_fr'] = ('GSM full-rate, 13 kbit/s (2G mobile)', gsm)
    if have_opus_tools:
        for p in (5, 10, 20):
            def f(x, u, p=p):
                y, lag = align_to(x, opus_plc(x, p, ffmpeg=ffmpeg))
                return y, {'lag': lag, 'lost': p / 100}
            C[f'opus_nb_8k_plc{p}'] = (f'Opus 8 kbit/s, {p}% packet loss, native Opus concealment', f)
    if have_webrtc and 'amrnb_12k2' in base:
        def apm_amr(x, u):
            w, lag = align_to(x, webrtc_apm(x))          # WebRTC adds a short delay: remove it first
            y, info = base['amrnb_12k2'][1](w, u)
            return y, dict(info, lag=lag + info.get('lag', 0))
        C['webrtc_amrnb_12k2'] = ('WebRTC noise suppression + gain control, then AMR-NB 12.2 kbit/s', apm_amr)
        def apm(x, u):
            y, lag = align_to(x, webrtc_apm(x))
            return y, {'lag': lag}
        C['webrtc_only'] = ('WebRTC noise suppression + gain control only', apm)
    if 'libopencore_amrnb' in encoders:                      # dose-response over AMR-NB rates
        for r in ('5.9', '7.4'):
            C[f"amrnb_{r.replace('.', 'k')}"] = (f'AMR-NB {r} kbit/s (mobile)', _coded(['-c:a', 'libopencore_amrnb', '-b:a', f'{r}k'], 'amr', 8000, ffmpeg))
    if 'libvo_amrwbenc' in encoders:                         # wideband (HD voice / VoLTE)
        for r in ('12.65', '23.85'):
            C[f"amrwb_{r.replace('.', 'k')}"] = (f'AMR-WB {r} kbit/s (HD voice, 16 kHz)', _coded(['-c:a', 'libvo_amrwbenc', '-b:a', f'{r}k'], 'amr', 16000, ffmpeg))
    if 'libopus' in encoders:                                # wideband VoIP, as WebRTC apps normally run
        for r in (16, 24):
            C[f'opus_wb_{r}k'] = (f'Opus wideband {r} kbit/s, VoIP mode', _coded(['-c:a', 'libopus', '-b:a', f'{r}k', '-application', 'voip'], 'opus', 16000, ffmpeg))
    if 'g711u' in base:                                      # bursty (Gilbert-Elliott) loss, mean burst 3 frames
        for m in (5, 10):
            def ge(x, u, m=m):
                y, info = base['g711u'][1](x, u)
                y, lost = frame_loss_ge(y, u, m / 100, 3.0, seed)
                return y, dict(info, lost=lost)
            C[f'g711u_burst{m}'] = (f'G.711 with {m}% bursty frame loss (Gilbert-Elliott, mean burst 60 ms)', ge)
    if 'amrnb_12k2' in base:
        def chain(x, u):
            y, info = base['amrnb_12k2'][1](bandpass(x), u)
            y, lost = frame_loss(y, u, 0.05, seed)
            return y, dict(info, lost=lost)
        C['chain_bp_amr12k2_loss5'] = ('Band-limit, AMR-NB 12.2 kbit/s, 5% frame loss', chain)
    return base, C


def shortcut_variants(stage_fn):
    """Loudness and silence controls applied AFTER the call stage (B in the mechanism slide).
    __rms : call copy loudness-normalised to -26 dBFS.
    __trim: clean-defined speech region cut from the call copy (leading/trailing silence removed)."""
    def rms(x, u):
        y, info = stage_fn(x, u)
        return rms_normalise(y), info
    def trim(x, u):
        y, info = stage_fn(x, u)
        s, e = trim_bounds(x)
        return y[s:e], dict(info, trim=(s, e))
    return rms, trim


def clean_trim(x, u):
    s, e = trim_bounds(x)
    return x[s:e], {'trim': (s, e)}


# ----------------------------- second architecture ------------------------
def build_aasist(repo_dir, device='cpu', variant='AASIST'):
    """Jung et al.'s released AASIST (raw-waveform, no SSL front end), trained on ASVspoof 2019 LA.
    Wrapped so model(x) returns logits like SSL-AASIST: column 1 = bona fide."""
    import torch, torch.nn as nn, importlib
    conf = json.load(open(f'{repo_dir}/config/{variant}.conf'))
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    mod = importlib.import_module('models.' + conf['model_config']['architecture'])
    net = mod.Model(conf['model_config'])
    sd = torch.load(f'{repo_dir}/models/weights/{variant}.pth', map_location='cpu')
    net.load_state_dict(sd, strict=True)

    class Wrap(nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        def forward(self, x):
            return self.m(x)[1]
    n_params = sum(p.numel() for p in net.parameters())
    return Wrap(net).to(device).eval(), {'arch': variant, 'params': n_params, 'nb_samp': conf['model_config']['nb_samp']}


# ----------------------------- inside the model ---------------------------
def xlsr_layers(net):
    """The 24 transformer layers of the (Hugging Face) XLS-R front end inside SSL-AASIST."""
    return net.ssl_model.model.encoder.layers


def _hs(out):
    return out[0] if isinstance(out, tuple) else out


def _replace(out, h):
    return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h


class LayerTaps:
    """Context manager: records the time-averaged output of every chosen layer (B x 1024 each)."""
    def __init__(self, net, layers):
        self.layers, self.store, self.handles = list(layers), {}, []
        self.mods = xlsr_layers(net)
    def __enter__(self):
        for l in self.layers:
            def hook(m, i, o, l=l):
                self.store[l] = _hs(o).float().mean(1).detach().cpu().numpy()
            self.handles.append(self.mods[l].register_forward_hook(hook))
        return self
    def __exit__(self, *a):
        for h in self.handles:
            h.remove()


class LayerShift:
    """Context manager: adds a fixed vector to every frame of one layer's output
    (test-time steering). vec: 1-D array of size 1024 (use -drift to remove a drift)."""
    def __init__(self, net, layer, vec):
        import torch
        self.mod = xlsr_layers(net)[layer]
        self.vec = torch.as_tensor(np.asarray(vec, np.float32))
        self.h = None
    def __enter__(self):
        def hook(m, i, o):
            h = _hs(o)
            return _replace(o, h + self.vec.to(h.device, h.dtype))
        self.h = self.mod.register_forward_hook(hook)
        return self
    def __exit__(self, *a):
        self.h.remove()


def speaker_split(speakers, seed=0):
    """Deterministic 50/50 split by speaker: 'fit' (estimate directions/thresholds) vs 'test'."""
    sp = sorted(set(speakers))
    rng = np.random.default_rng(seed); rng.shuffle(sp)
    fit = set(sp[:len(sp) // 2])
    return np.array(['fit' if s in fit else 'test' for s in speakers])



class LayerAffine:
    """Context manager: replaces one layer's output h (every frame) by h @ A + b (label-free alignment)."""
    def __init__(self, net, layer, A, b):
        import torch
        self.mod = xlsr_layers(net)[layer]
        self.A = torch.as_tensor(np.asarray(A, np.float32)); self.b = torch.as_tensor(np.asarray(b, np.float32))
        self.h = None
    def __enter__(self):
        def hook(m, i, o):
            h = _hs(o)
            return _replace(o, h @ self.A.to(h.device, h.dtype) + self.b.to(h.device, h.dtype))
        self.h = self.mod.register_forward_hook(hook)
        return self
    def __exit__(self, *a):
        self.h.remove()


def coral_affine(src, tgt, reg=0.1):
    """CORAL (Sun et al., 2016) as an affine map from the call-copy statistics (src) to the clean statistics
    (tgt), both N x D utterance means from the fit half. Returns A (D x D) and b (D) with x' = x @ A + b.
    reg = shrinkage towards the diagonal, since N is close to D."""
    def shrink(C):
        return (1 - reg) * C + reg * np.diag(np.diag(C)) + 1e-6 * np.eye(len(C))
    def msqrt(C, inv=False):
        w, V = np.linalg.eigh(C); w = np.clip(w, 1e-8, None)
        return (V * (w ** (-0.5 if inv else 0.5))) @ V.T
    ms, mt = src.mean(0), tgt.mean(0)
    Cs, Ct = shrink(np.cov(src.T)), shrink(np.cov(tgt.T))
    A = msqrt(Cs, inv=True) @ msqrt(Ct)
    return A.astype(np.float32), (mt - ms @ A).astype(np.float32)

In [ ]:
#@title 6 · Stages, detectors, thresholds and the scorer (resumes from Drive)
import sys, importlib, torch, soundfile as sf, gc
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '/content'); import pilot_lib as P; importlib.reload(P)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'Switch the runtime to a T4 GPU first.'

BASE, NEW = P.make_conditions_v2(enc, CFG['SEED'], FFMPEG, FFMPEG_GSM, HAVE_OPUS_TOOLS, HAVE_WEBRTC)
DAMAGING = [c for c in ('g711u_loss10', 'amrnb_4k75', 'opus_nb_8k') if c in BASE]
SHORT = {'clean_trim': ('Clean, leading/trailing silence removed (reference for __trim)', P.clean_trim)}
for c in DAMAGING:
    r, t = P.shortcut_variants(BASE[c][1])
    SHORT[f'{c}__rms'] = (f'{c}, then loudness-normalised to -26 dBFS', r)
    SHORT[f'{c}__trim'] = (f'{c}, then leading/trailing silence removed', t)
ALL = {**BASE, **NEW, **SHORT}
# check every stage on 2 s of noise; drop any whose codec is missing in this ffmpeg build
_x = (0.05 * np.random.default_rng(0).standard_normal(32000)).astype(np.float32)
for k in list(ALL):
    try:
        y, _ = ALL[k][1](_x, 'probe'); assert len(y) > 0
    except Exception as e:
        print(f'  dropping {k}: {str(e)[:120]}'); ALL.pop(k)
print(f'{len(ALL)} stages:'); [print(f'  {k:26s} {v[0]}') for k, v in ALL.items()]

FOLDER = {'A': f"{CFG['OUT']}/scores", 'B': f"{CFG['OUT']}/run_LA_model/scores", 'AASIST': f"{CFG['OUT']}/run_AASIST/scores"}
NAME = {'A': 'SSL-AASIST A (DF-tuned RawBoost)', 'B': 'SSL-AASIST B (LA-tuned RawBoost)', 'AASIST': 'AASIST (no SSL front end)'}
for f in FOLDER.values(): os.makedirs(f, exist_ok=True)
LOAD = {}

def load_detector(k):
    m, rep = P.build_aasist(AREPO, dev) if k == 'AASIST' else P.build_ssl_aasist(REPO, CK[k], dev)
    LOAD[k] = rep
    return m

def prep(row, cond):
    x, sr = sf.read(row.path, dtype='float32'); assert sr == P.SR
    y, info = ALL[cond][1](x, row.utt)
    return P.tak_pad(y), info

@torch.no_grad()
def score_set(model, df, cond, folder):
    out = f'{folder}/{df.part.iloc[0]}__{cond}.csv'
    if os.path.exists(out):
        d = pd.read_csv(out)
        if 'logit0' in d.columns or not CFG['RESCORE_FOR_LLR'] or model is None:
            return d
    t0, rows = time.time(), []
    with ThreadPoolExecutor(4) as ex:
        items = list(df.itertuples())
        for i in range(0, len(items), CFG['BATCH']):
            chunk = items[i:i + CFG['BATCH']]
            res = list(ex.map(lambda r: prep(r, cond), chunk))
            o = model(torch.tensor(np.stack([r[0] for r in res])).to(dev)).float().cpu().numpy()
            for r, (_, info), v in zip(chunk, res, o):
                rows.append(dict(utt=r.utt, speaker=r.speaker, attack=r.attack, label=r.label, cond=cond,
                                 score=float(v[1]), logit0=float(v[0]), lag=info.get('lag', 0), lost=info.get('lost', 0.0)))
    d = pd.DataFrame(rows); d.to_csv(out, index=False)
    print(f'  {df.part.iloc[0]:4s} {cond:26s} {len(d)} utts in {time.time()-t0:.0f}s')
    return d

TH = {}
def thresholds(k, model=None):
    # dev clean scores set every threshold: dev EER point (main), centre of the clean gap, 1% of dev fakes accepted
    dv = score_set(model, DV, 'clean', FOLDER[k]) if model is not None else pd.read_csv(f'{FOLDER[k]}/dev__clean.csv')
    b, s = dv[dv.label == 1].score.values, dv[dv.label == 0].score.values
    TH[k] = {'dev_EER': P.eer(b, s)[1], 'mid_gap': (np.percentile(b, 1) + np.percentile(s, 99)) / 2, 'fakes_1pct': np.percentile(s, 99)}
    return P.eer(b, s)[0], TH[k]

In [ ]:
#@title 7 · Inside the model: record the 24 XLS-R layers, clean vs call (about 10 min per checkpoint)
LAYERS_ALL = list(range(24))
SPLIT = P.speaker_split(EV.speaker.values, CFG['SEED'])        # half the speakers fit, the other half test (speaker-disjoint)
print('fit / test utterances:', (SPLIT == 'fit').sum(), (SPLIT == 'test').sum())
INSIDE_CONDS = ['clean'] + DAMAGING
WAV = {}                                                        # processed audio per half, reused by step 9

def waves(cond, part='test'):
    if (cond, part) not in WAV:
        idx = np.where(SPLIT == part)[0]
        with ThreadPoolExecutor(4) as ex:
            WAV[(cond, part)] = np.stack(list(ex.map(lambda i: prep(EV.iloc[i], cond)[0], idx))).astype(np.float32)
    return WAV[(cond, part)]

@torch.no_grad()
def record(model, cond, path):
    if os.path.exists(path):
        z = np.load(path); return z['H'].astype(np.float32), z['score']
    H, S = np.zeros((len(EV), 24, 1024), np.float16), np.zeros(len(EV), np.float32)
    with ThreadPoolExecutor(4) as ex:
        for i in range(0, len(EV), CFG['BATCH']):
            rows = [EV.iloc[j] for j in range(i, min(i + CFG['BATCH'], len(EV)))]
            x = np.stack([r[0] for r in ex.map(lambda r: prep(r, cond), rows)])
            with P.LayerTaps(model, LAYERS_ALL) as tp:
                s = model(torch.tensor(x).to(dev))[:, 1].float().cpu().numpy()
            H[i:i + len(rows)] = np.stack([tp.store[l] for l in LAYERS_ALL], 1).astype(np.float16)
            S[i:i + len(rows)] = s
    np.savez(path, H=H, score=S, utt=EV.utt.values)
    return H.astype(np.float32), S

INSIDE = {}
for k in CFG['INSIDE_CKPTS']:
    d = f"{CFG['OUT']}/week2/inside_{k}"; os.makedirs(d, exist_ok=True)
    need = [c for c in INSIDE_CONDS if not os.path.exists(f'{d}/layers_{c}.npz')] or not os.path.exists(f'{FOLDER[k]}/dev__clean.csv')
    model = load_detector(k) if need else None
    thresholds(k, model)
    INSIDE[k] = {}
    for c in INSIDE_CONDS:
        t0 = time.time(); INSIDE[k][c] = record(model, c, f'{d}/layers_{c}.npz'); print(f'{k} {c:14s} recorded in {time.time()-t0:.0f}s')
    if model is not None:
        del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
#@title 8 · Where does the call push genuine speech toward "fake"? Can a probe still tell them apart? (CPU, ~3 min)
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
y = EV.label.values; fit, test = SPLIT == 'fit', SPLIT == 'test'
GEOM, PROBE = [], []
for k in CFG['INSIDE_CKPTS']:
    Hc = INSIDE[k]['clean'][0]
    u = Hc[fit & (y == 0)].mean(0) - Hc[fit & (y == 1)].mean(0)          # real -> fake axis on clean audio, per layer
    for c in DAMAGING:
        Hx = INSIDE[k][c][0]; D = Hx - Hc
        for l in LAYERS_ALL:
            ul = u[l]; g = ul @ ul
            vb, vs = D[fit & (y == 1), l].mean(0), D[fit & (y == 0), l].mean(0)
            GEOM.append(dict(ckpt=k, stage=c, layer=l, genuine_moves=float(vb @ ul / g), fake_moves=float(vs @ ul / g),
                             cos_genuine=float(vb @ ul / (np.linalg.norm(vb) * np.sqrt(g) + 1e-9)),
                             drift_norm_ratio=float(np.linalg.norm(vb) / np.sqrt(g))))
        for l in LAYERS_ALL:
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000)).fit(Hc[fit, l], y[fit])
            clf2 = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000)).fit(Hx[fit, l], y[fit])
            sc_call = clf.decision_function(Hx[test, l])
            PROBE.append(dict(ckpt=k, stage=c, layer=l,
                              auc_clean=roc_auc_score(y[test], clf.decision_function(Hc[test, l])),
                              auc_call_clean_probe=roc_auc_score(y[test], sc_call),
                              eer_call_clean_probe=P.eer(sc_call[y[test] == 1], sc_call[y[test] == 0])[0],
                              auc_call_retrained=roc_auc_score(y[test], clf2.decision_function(Hx[test, l]))))
GEOM, PROBE = pd.DataFrame(GEOM), pd.DataFrame(PROBE)
GEOM.to_csv(f"{CFG['OUT']}/week2/inside_geometry.csv", index=False); PROBE.to_csv(f"{CFG['OUT']}/week2/inside_probes.csv", index=False)

for k in CFG['INSIDE_CKPTS']:
    fig, axes = plt.subplots(1, len(DAMAGING), figsize=(4.2 * len(DAMAGING), 3.4), sharey=True)
    for ax, c in zip(np.atleast_1d(axes), DAMAGING):
        g = GEOM[(GEOM.ckpt == k) & (GEOM.stage == c)]
        ax.plot(g.layer, g.genuine_moves, color='#B04A17', lw=2, label='genuine speech')
        ax.plot(g.layer, g.fake_moves, color='#6B7A80', lw=2, label='fakes')
        ax.axhline(0, color='#999', lw=.8); ax.axhline(1, color='#999', lw=.8, ls='--')
        ax.set_title(c); ax.set_xlabel('XLS-R layer')
    np.atleast_1d(axes)[0].set_ylabel('share of the real→fake gap\nmoved by the call'); np.atleast_1d(axes)[0].legend(frameon=False)
    fig.suptitle(f'Checkpoint {k}: where the call moves each class along the clean real→fake axis (1 = all the way)')
    fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/figures/week2_inside_drift_{k}.png", dpi=200); plt.show()

display(GEOM[GEOM.layer.isin([0, 5, 11, 17, 23])].pivot_table(index=['ckpt', 'stage'], columns='layer', values=['genuine_moves', 'fake_moves']).round(2))
display(PROBE[PROBE.layer.isin([0, 5, 11, 17, 23])].set_index(['ckpt', 'stage', 'layer']).round(3))
print('Reading: genuine_moves large and fake_moves near 0 = the call pushes real voices toward fake inside the model.')
print('auc_call_retrained near 1 while the original detector fails = the information survives the call but is misread, not lost.')

In [ ]:
#@title 9 · A first training-free fix at one layer, against the baselines (about 45-60 min; AMR-NB 4.75 first)
# Fitted on the 'fit' speakers, tested on the other half. 'fitted on other codecs' never sees the codec it is tested on.
FIX = f"{CFG['OUT']}/week2/test_time_fix.csv"
done = pd.read_csv(FIX) if os.path.exists(FIX) else pd.DataFrame(columns=['ckpt', 'stage', 'layer', 'variant'])
have = {(r['ckpt'], r['stage'], int(r['layer']), r['variant']) for _, r in done.iterrows()}
out_rows = done.to_dict('records')
yt = y[test]

def metrics(s, thr):
    b, f = s[yt == 1], s[yt == 0]
    fa, miss = P.rates_at(b, f, thr)
    return dict(eer=P.eer(b, f)[0], real_rejected=fa, fakes_accepted=miss, genuine_mean=float(b.mean()), fake_mean=float(f.mean()))

@torch.no_grad()
def score_with(model, X, ctx=None):
    out = []
    for i in range(0, len(X), CFG['BATCH']):
        x = torch.tensor(X[i:i + CFG['BATCH']]).to(dev)
        if ctx is None:
            s = model(x)
        else:
            with ctx():
                s = model(x)
        out.append(s[:, 1].float().cpu().numpy())
    return np.concatenate(out)

def add(row):
    out_rows.append(row); pd.DataFrame(out_rows).to_csv(FIX, index=False)

import copy
@torch.no_grad()
def bn_adapt_scores(model, cond):
    # Zhou & Wang's BatchNorm adaptation: re-estimate the back end's BatchNorm statistics on the fit half's
    # call audio (no labels), score the test half, then restore the original weights.
    state = copy.deepcopy(model.state_dict())
    bns = [m for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
    for m in bns:
        m.reset_running_stats(); m.momentum = None; m.train()
    Xf = waves(cond, 'fit')
    for i in range(0, len(Xf), CFG['BATCH']):
        model(torch.tensor(Xf[i:i + CFG['BATCH']]).to(dev))
    for m in bns:
        m.eval()
    s = score_with(model, waves(cond))
    model.load_state_dict(state)
    for m in bns:
        m.momentum = 0.1
    return s, len(bns)

def subspace_affine(Hx_fit, Hc_fit, k):
    # remove the top-k directions of the per-utterance drift (SVD), and align the mean to clean
    D = Hx_fit - Hc_fit
    U = np.linalg.svd(D, full_matrices=False)[2][:k].T                # 1024 x k
    A = np.eye(D.shape[1], dtype=np.float32) - U @ U.T
    return A.astype(np.float32), (Hc_fit.mean(0) - Hx_fit.mean(0) @ A).astype(np.float32)

rng = np.random.default_rng(CFG['SEED'])
for k in CFG['INSIDE_CKPTS']:
    model, THR = None, TH[k]['dev_EER']
    Hc, Sc = INSIDE[k]['clean']
    if (k, 'clean', -1, 'none') not in have:
        add(dict(ckpt=k, stage='clean', layer=-1, variant='none', **metrics(Sc[test], THR)))
    for c in sorted(DAMAGING, key=lambda z: z != 'amrnb_4k75'):        # AMR-NB 4.75 first: the key result
        Hx, Sx = INSIDE[k][c]
        if (k, c, -1, 'BatchNorm adaptation on call audio (label-free)') not in have:
            if model is None:
                model = load_detector(k)
            s_bn, n_bn = bn_adapt_scores(model, c)
            add(dict(ckpt=k, stage=c, layer=-1, variant='BatchNorm adaptation on call audio (label-free)', **metrics(s_bn, THR)))
            print(f'{k} {c}: BatchNorm adaptation ({n_bn} BN layers) EER {100*out_rows[-1]["eer"]:.1f}%')
        base = [('none', metrics(Sx[test], THR)),
                ('recalibrate threshold (labelled call data)', metrics(Sx[test], P.eer(Sx[fit & (y == 1)], Sx[fit & (y == 0)])[1])),
                ('shift threshold by mean genuine drift', metrics(Sx[test], THR + float((Sx - Sc)[fit & (y == 1)].mean())))]
        for v, m in base:
            if (k, c, -1, v) not in have:
                add(dict(ckpt=k, stage=c, layer=-1, variant=v, **m))
        for l in CFG['INSIDE_LAYERS']:
            D = (Hx - Hc)[:, l]
            v_all = D[fit].mean(0)                                   # label-free: clean/call pairs only (= layer mean alignment)
            v_bona = D[fit & (y == 1)].mean(0)                       # uses genuine labels
            others = [o for o in DAMAGING if o != c]
            v_held = np.mean([(INSIDE[k][o][0] - Hc)[fit, l].mean(0) for o in others], 0) if others else v_all
            r = rng.standard_normal(1024).astype(np.float32); r *= np.linalg.norm(v_all) / np.linalg.norm(r)
            A, b = P.coral_affine(Hx[fit, l], Hc[fit, l])
            A4, b4 = subspace_affine(Hx[fit, l], Hc[fit, l], 4)
            A16, b16 = subspace_affine(Hx[fit, l], Hc[fit, l], 16)
            variants = [
                ('remove drift, label-free (layer mean alignment)', c, lambda: P.LayerShift(model, l, -v_all)),
                ('remove drift, genuine pairs only', c, lambda: P.LayerShift(model, l, -v_bona)),
                ('remove drift fitted on OTHER codecs', c, lambda: P.LayerShift(model, l, -v_held)),
                ('CORAL at this layer, label-free', c, lambda: P.LayerAffine(model, l, A, b)),
                ('remove top-4 drift directions, label-free', c, lambda: P.LayerAffine(model, l, A4, b4)),
                ('remove top-16 drift directions, label-free', c, lambda: P.LayerAffine(model, l, A16, b16)),
                ('random direction, same size', c, lambda: P.LayerShift(model, l, -r)),
                ('clean cost: label-free fix applied to clean audio', 'clean', lambda: P.LayerShift(model, l, -v_all)),
                ('reverse patch: add the drift to clean audio', 'clean', lambda: P.LayerShift(model, l, v_all)),
                ('reverse patch control: add a random direction to clean audio', 'clean', lambda: P.LayerShift(model, l, r)),
            ]
            for name, cond, ctx in variants:
                if (k, c, l, name) in have:
                    continue
                if model is None:
                    model = load_detector(k)
                t0 = time.time()
                m = metrics(score_with(model, waves(cond), ctx), THR)
                add(dict(ckpt=k, stage=c, layer=l, variant=name, **m))
                print(f'{k} {c:13s} L{l:2d} {name:50s} EER {100*m["eer"]:5.1f}%  real rej {100*m["real_rejected"]:5.1f}%  fakes acc {100*m["fakes_accepted"]:5.1f}%  ({time.time()-t0:.0f}s)')
    if model is not None:
        del model; gc.collect(); torch.cuda.empty_cache()
FIXT = pd.DataFrame(out_rows); FIXT.to_csv(FIX, index=False)
display((FIXT.assign(**{m: 100 * FIXT[m] for m in ('eer', 'real_rejected', 'fakes_accepted')})
         .pivot_table(index=['ckpt', 'stage', 'variant'], columns='layer', values=['eer', 'real_rejected', 'fakes_accepted']).round(1)))
print('Success looks like: lower EER and real callers rejected than "none" AND than the recalibration rows, with the random direction')
print('doing nothing, the OTHER-codecs version still helping, small clean cost, and the reverse patch raising real callers rejected.')

In [ ]:
#@title 10 · Score every detector on every stage (about 1.5-2 hours; resumes)
FIRST = [c for c in ALL if c in NEW or c in SHORT or c not in BASE]          # new stages first
ORDER_RUN = ['clean'] + [c for c in FIRST if c != 'clean'] + [c for c in BASE if c not in FIRST and c != 'clean']
for k in CFG['RUN_DETECTORS']:
    def need(c):
        f = f'{FOLDER[k]}/eval__{c}.csv'
        return not os.path.exists(f) or (CFG['RESCORE_FOR_LLR'] and 'logit0' not in pd.read_csv(f, nrows=1).columns)
    todo = [c for c in ORDER_RUN if need(c)]
    print(f'\n== {NAME[k]}: {len(todo)} stages to score')
    if not todo and os.path.exists(f'{FOLDER[k]}/dev__clean.csv'):
        continue
    model = load_detector(k)
    print('loaded:', LOAD[k])
    thresholds(k, model)
    for c in ORDER_RUN:
        score_set(model, EV, c, FOLDER[k])
    del model; gc.collect(); torch.cuda.empty_cache()
print('\nAll scoring done.')

In [ ]:
#@title 11 · Results: every detector, the shortcut test, 95% CIs on EER changes, threshold robustness, per attack
def boot_delta_eer(d, ref, n=300, seed=0):
    # paired bootstrap over utterances: EER(stage) - EER(reference), 95% CI
    rng = np.random.default_rng(seed); b = d.label.values == 1
    ds, rs, out = d.score.values, ref.score.values, []
    ib, isp = np.where(b)[0], np.where(~b)[0]
    for _ in range(n):
        jb, js = rng.choice(ib, ib.size), rng.choice(isp, isp.size)
        out.append(P.eer(ds[jb], ds[js])[0] - P.eer(rs[jb], rs[js])[0])
    return np.percentile(out, 2.5), np.percentile(out, 97.5)

rows, thr_rows, att_rows, S_ALL = [], [], [], {}
for k in CFG['RUN_DETECTORS']:
    dev_eer, _ = thresholds(k); THR = TH[k]['dev_EER']
    S = {c: pd.read_csv(f'{FOLDER[k]}/eval__{c}.csv').set_index('utt') for c in ALL if os.path.exists(f'{FOLDER[k]}/eval__{c}.csv')}
    S_ALL[k] = S
    for c, d in S.items():
        rname = 'clean_trim' if c.endswith('__trim') and 'clean_trim' in S else 'clean'
        ref = S[rname].loc[d.index].copy(); ref['correct_clean'] = (ref.score >= THR) == (ref.label == 1)
        d = d.copy(); d['correct_clean'] = ref.correct_clean
        r = P.summarise(d, ref, THR)['all']
        rc = P.summarise(ref, ref, THR)['all']
        counts, kind, d_eer, d_fa = P.decision(rc, r)
        lo, hi = boot_delta_eer(d, ref) if c not in ('clean', 'clean_trim') else (0.0, 0.0)
        row = dict(detector=k, condition=c, reference=rname, dev_eer=dev_eer, **r,
                   kind=kind if c not in ('clean', 'clean_trim') else '', delta_eer=d_eer, delta_eer_lo=lo, delta_eer_hi=hi, delta_fa=d_fa)
        if 'logit0' in d.columns and 'logit0' in ref.columns:
            llr, llr0 = d.score - d.logit0, ref.score - ref.logit0
            row['llr_shift_bona'] = float((llr - llr0)[d.label == 1].mean()); row['llr_shift_spoof'] = float((llr - llr0)[d.label == 0].mean())
        rows.append(row)
        b, s = d[d.label == 1].score.values, d[d.label == 0].score.values
        for name, t in TH[k].items():
            fa, miss = P.rates_at(b, s, t)
            thr_rows.append(dict(detector=k, condition=c, threshold=name, real_rejected=fa, fakes_accepted=miss))
        if c in ['clean'] + DAMAGING:
            for a, g in d[d.label == 0].groupby('attack'):
                att_rows.append(dict(detector=k, condition=c, attack=a, fakes_accepted=float(np.mean(g.score >= THR))))
R = pd.DataFrame(rows); R.to_csv(f"{CFG['OUT']}/week2/week2_results.csv", index=False)
T = pd.DataFrame(thr_rows); T.to_csv(f"{CFG['OUT']}/week2/week2_threshold_robustness.csv", index=False)
ATT = pd.DataFrame(att_rows); ATT.to_csv(f"{CFG['OUT']}/week2/week2_per_attack.csv", index=False)

ORDER = ['clean', 'clean_rms', 'clean_trim', 'bandpass_300_3400', 'nb_resample', 'g711u', 'g711u_burst5', 'g711u_burst10', 'g711u_loss10',
         'gsm_fr', 'amrwb_23k85', 'amrwb_12k65', 'amrnb_12k2', 'amrnb_7k4', 'amrnb_5k9', 'amrnb_4k75', 'opus_wb_24k', 'opus_wb_16k',
         'opus_nb_8k', 'opus_nb_8k_plc5', 'opus_nb_8k_plc10', 'opus_nb_8k_plc20', 'webrtc_only', 'webrtc_amrnb_12k2',
         'chain_bp_amr12k2_loss5'] + [f'{c}__{v}' for c in DAMAGING for v in ('rms', 'trim')]
def table(k):
    t = R[R.detector == k].set_index('condition')
    t = t.reindex([c for c in ORDER if c in t.index] + [c for c in t.index if c not in ORDER])
    out = pd.DataFrame({'EER %': 100 * t.eer,
                        'ΔEER [95% CI]': [f'{100*a:+.1f} [{100*lo:+.1f}, {100*hi:+.1f}]' for a, lo, hi in zip(t.delta_eer, t.delta_eer_lo, t.delta_eer_hi)],
                        'real rejected %': 100 * t.fa_bona_at_thr, 'fakes accepted %': 100 * t.miss_spoof_at_thr,
                        'actDCF': t.actDCF_asv5, 'minDCF': t.minDCF_asv5,
                        'genuine shift [95% CI]': [f'{m:+.2f} [{lo:+.2f}, {hi:+.2f}]' for m, lo, hi in zip(t.shift_bona_mean, t.shift_bona_lo, t.shift_bona_hi)],
                        'fake shift [95% CI]': [f'{m:+.2f} [{lo:+.2f}, {hi:+.2f}]' for m, lo, hi in zip(t.shift_spoof_mean, t.shift_spoof_lo, t.shift_spoof_hi)],
                        'kind': t.kind})
    if 'llr_shift_bona' in t.columns:
        out['LLR shift genuine / fake'] = [f'{a:+.2f} / {b:+.2f}' for a, b in zip(t.llr_shift_bona, t.llr_shift_spoof)]
    return out
for k in CFG['RUN_DETECTORS']:
    print(f'\n=== {NAME[k]}  (dev EER {100*R[R.detector==k].dev_eer.iloc[0]:.2f}%, clean eval EER {100*R[(R.detector==k)&(R.condition=="clean")].eer.iloc[0]:.2f}%)')
    display(table(k).round(3))

print('\n=== Shortcut test: does loudness normalisation or silence removal undo the damage? (real callers rejected % / EER %)')
sc = []
for k in CFG['RUN_DETECTORS']:
    t = R[R.detector == k].set_index('condition')
    for c in DAMAGING:
        row = {'detector': k, 'stage': c}
        for suf, lab in (('', 'as is'), ('__rms', '+ loudness norm'), ('__trim', '+ silence removed')):
            if c + suf in t.index:
                row[lab] = f"{100*t.loc[c+suf, 'fa_bona_at_thr']:.1f} / {100*t.loc[c+suf, 'eer']:.1f}"
        sc.append(row)
SC_T = pd.DataFrame(sc); display(SC_T)
print('If "+ loudness norm" or "+ silence removed" brings real callers rejected back near 0, the shortcut explains the failure.')
print('\n=== Fakes accepted % per attack at the dev-EER threshold')
display((100 * ATT.pivot_table(index=['detector', 'attack'], columns='condition', values='fakes_accepted')).round(1))

In [ ]:
#@title 12 · Ceiling check: do fakes "not move" only because they are already at the bottom? (CPU)
CEIL = []
for k in CFG['RUN_DETECTORS']:
    S = S_ALL[k]; c0 = S['clean']; THR = TH[k]['dev_EER']
    for c in DAMAGING:
        if c not in S: continue
        d = S[c].loc[c0.index]; sh = d.score - c0.score
        sp = c0[c0.label == 0]; hard = sp.score >= sp.score.quantile(0.9)          # the 10% of fakes closest to "genuine" when clean
        margin = (c0.score - THR).abs()
        row = dict(detector=k, stage=c,
                   genuine_shift=float(sh[c0.label == 1].mean()), fake_shift=float(sh[c0.label == 0].mean()),
                   hardest_fakes_shift=float(sh[sp.index[hard.values]].mean()),
                   genuine_shift_per_margin=float((sh / (margin + 1e-6))[c0.label == 1].median()),
                   fake_shift_per_margin=float((sh / (margin + 1e-6))[c0.label == 0].median()),
                   corr_genuine_shift_vs_clean_score=float(np.corrcoef(c0.score[c0.label == 1], sh[c0.label == 1])[0, 1]))
        if 'logit0' in d.columns and 'logit0' in c0.columns:
            l1, l0 = d.score - d.logit0, c0.score - c0.logit0
            row['llr_genuine_shift'] = float((l1 - l0)[c0.label == 1].mean()); row['llr_fake_shift'] = float((l1 - l0)[c0.label == 0].mean())
            row['fake_logit_shift_genuine'] = float((d.logit0 - c0.logit0)[c0.label == 1].mean())
        CEIL.append(row)
CEIL = pd.DataFrame(CEIL); CEIL.to_csv(f"{CFG['OUT']}/week2/week2_ceiling_check.csv", index=False); display(CEIL.round(3))
print('If the hardest fakes (already near the boundary) also barely move, and the LLR shows the same one-sided drift,')
print('the asymmetry is not just a floor effect.')

In [ ]:
#@title 13 · Optional: modern commercial fakes (ElevenLabs / Respeecher), fakes accepted
from scipy.signal import resample_poly
ELEV = None
if CFG['ELEVEN_DIR'] and os.path.isdir(CFG['ELEVEN_DIR']):
    files = sorted(glob.glob(f"{CFG['ELEVEN_DIR']}/**/*.wav", recursive=True) + glob.glob(f"{CFG['ELEVEN_DIR']}/**/*.mp3", recursive=True))
    print(len(files), 'clips found')
    out = f"{CFG['OUT']}/week2/elevenlabs_scores.csv"
    if os.path.exists(out):
        ELEV = pd.read_csv(out)
    else:
        def load16(p):
            x, sr = sf.read(p, dtype='float32', always_2d=True); x = x[:, 0]
            if sr != P.SR:
                g = np.gcd(sr, P.SR); x = resample_poly(x, P.SR // g, sr // g).astype(np.float32)
            return x
        rows = []
        for k in CFG['RUN_DETECTORS']:
            model = load_detector(k)
            for c in ['clean'] + DAMAGING:
                for i in range(0, len(files), CFG['BATCH']):
                    chunk = files[i:i + CFG['BATCH']]
                    xs = [P.tak_pad(ALL[c][1](load16(p), os.path.basename(p))[0]) for p in chunk]
                    with torch.no_grad():
                        s = model(torch.tensor(np.stack(xs)).to(dev))[:, 1].float().cpu().numpy()
                    rows += [dict(detector=k, cond=c, file=os.path.basename(p), tool='ElevenLabs' if 'eleven' in p.lower() else ('Respeecher' if 'respeech' in p.lower() else 'unknown'), score=float(v)) for p, v in zip(chunk, s)]
                print(k, c, 'done')
            del model; gc.collect(); torch.cuda.empty_cache()
        ELEV = pd.DataFrame(rows); ELEV.to_csv(out, index=False)
    acc = []
    for (k, c), g in ELEV.groupby(['detector', 'cond']):
        acc.append({'detector': k, 'stage': c, **{f'accepted @ {n} %': 100 * np.mean(g.score >= t) for n, t in TH[k].items()}})
    ELEV_T = pd.DataFrame(acc); display(ELEV_T.round(1))
    print('High "accepted" on clean = the detector misses modern fakes even before any call (generator shift, not the channel).')
else:
    print('Skipped (set CFG["ELEVEN_DIR"] to the unzipped Mendeley folder to run it).')

In [ ]:
#@title 14 · Summary to send to Claude (copy everything in the box)
import platform, transformers, io
buf = io.StringIO()
def w(*a): print(*a, file=buf)
w('### WEEK 2 SUMMARY (v2)'); w('run:', time.strftime('%Y-%m-%d %H:%M'), '| GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__,
  '| transformers', transformers.__version__, '| python', platform.python_version()); w('commits:', COMMITS)
w('ffmpeg:', FFMPEG, 'enc:', sorted(enc), 'opus-tools:', HAVE_OPUS_TOOLS, 'webrtc:', HAVE_WEBRTC); w('stages:', list(ALL))
w('loads:', LOAD); w('thresholds:', {k: {n: round(float(t), 3) for n, t in v.items()} for k, v in TH.items()})
if len(CFG['INSIDE_CKPTS']):
    w('\n## Inside: share of real->fake gap moved (genuine / fake) by layer')
    w(GEOM.pivot_table(index=['ckpt', 'stage'], columns='layer', values=['genuine_moves', 'fake_moves']).round(2).to_csv(sep='|'))
    w('\n## Inside: probes by layer'); w(PROBE.round(3).to_csv(sep='|', index=False))
    w('\n## Test-time fix (test half; percentages)')
    w((FIXT.assign(**{m: 100 * FIXT[m] for m in ('eer', 'real_rejected', 'fakes_accepted')})).round(2).to_csv(sep='|', index=False))
for k in CFG['RUN_DETECTORS']:
    if (R.detector == k).any():
        w(f'\n## {NAME[k]}'); w(table(k).round(3).to_csv(sep='|'))
w('\n## Shortcut test (real rejected % / EER %)'); w(SC_T.to_csv(sep='|', index=False))
w('\n## Ceiling check'); w(CEIL.round(3).to_csv(sep='|', index=False))
w('\n## Threshold robustness (real rejected / fakes accepted %, at dev_EER, mid_gap, fakes_1pct)')
TT = T.pivot_table(index=['detector', 'condition'], columns='threshold', values=['real_rejected', 'fakes_accepted'])
w((100 * TT).round(1).to_csv(sep='|'))
w('\n## Fakes accepted % per attack'); w((100 * ATT.pivot_table(index=['detector', 'attack'], columns='condition', values='fakes_accepted')).round(1).to_csv(sep='|'))
if ELEV is not None:
    w('\n## ElevenLabs/Respeecher fakes accepted %'); w(ELEV_T.round(1).to_csv(sep='|', index=False))
s = buf.getvalue()
open(f"{CFG['OUT']}/week2/week2_summary.txt", 'w').write(s)
json.dump(dict(cfg=CFG, commits=COMMITS, loads={k: str(v) for k, v in LOAD.items()}, ffmpeg=FFMPEG, encoders=sorted(enc), stages=list(ALL)),
          open(f"{CFG['OUT']}/week2/run_log.json", 'w'), indent=1, default=str)
print(s)
print('\nSaved to Drive: fyp_pilot/week2/week2_summary.txt  (send that file, or copy this box)')
print('Figure: fyp_pilot/figures/week2_inside_drift_A.png')